# Associate Fink Alerts with Rubin Visit Metadata

**Purpose:**  
Merge (left-join) each row of the Fink alert tables (`gaia_star_stable_src.parquet` and  
`gaia_star_stable_fp.parquet`) with Rubin visit metadata extracted at USDF.  
The join key is the **13-digit visit identifier**: `r:visit` in the alert table,  
matched against either:
- **Butler visitTable** (`visitTable-*_WithTracts.parquet`) — column `id`
- **consDb visitTable** (`constDbVisitTable-*_WithTracts.parquet`) — column `visit_id`

The output enriched tables are written to `data_FINK_BLOCK_LC_03/`.

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Creation Date:** 2026-04-01  
**Last update:** 2026-03-02
**Last update:** 2026-04-07
**Notebook tag:** `FINK_BLOCK_LC_03`

## Inputs

| File | Location | Description |
|------|----------|-------------|
| `gaia_star_stable_src.parquet` | `data_FINK_BLOCK_LC_01/` | Fink DIA sources (detections) |
| `gaia_star_stable_fp.parquet` | `data_FINK_BLOCK_LC_01/` | Fink forced photometry rows |
| `visitTable-*_WithTracts.parquet` | `../05_runbindata_visits/data_fromlsst/` | Butler visit table (N≈52 k) |
| `constDbVisitTable-*_WithTracts.parquet` | `../05_runbindata_visits/data_fromlsst/` | consDb visit table (N≈85 k) |

## Join key

```
alert["r:visit"]  ←→  butler["id"]      (Butler visitTable)
alert["r:visit"]  ←→  consdb["visit_id"] (consDb visitTable)
```

Both sides are cast to `int64` before joining.

## Outputs  (`data_FINK_BLOCK_LC_03/`)

| File | Description |
|------|-------------|
| `src_joined_butler.parquet` | src alerts + Butler visit columns |
| `src_joined_consdb.parquet` | src alerts + consDb visit columns |
| `fp_joined_butler.parquet`  | fp  alerts + Butler visit columns |
| `fp_joined_consdb.parquet`  | fp  alerts + consDb visit columns |


## 0. Imports & configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
!ls -l ../05_runbindata_visits/data_fromlsst/

In [ ]:
# ── Input paths ───────────────────────────────────────────────────────────────
NB_TAG_IN = "FINK_BLOCK_LC_01"
DIR_DATA_IN = f"data_{NB_TAG_IN}"  # relative to this notebook

FILE_SRC = os.path.join(DIR_DATA_IN, "gaia_star_stable_src.parquet")
FILE_FP = os.path.join(DIR_DATA_IN, "gaia_star_stable_fp.parquet")

# ── Rubin visit tables (extracted at USDF) ────────────────────────────────────
DIR_VISITS = os.path.join("..", "05_runbindata_visits", "data_fromlsst")

# FILE_BUTLER = os.path.join(DIR_VISITS, "visitTable-2025041500138-2026031900284_N52726_WithTracts.parquet")
FILE_BUTLER = os.path.join(DIR_VISITS, "visitTable-2025041500138-2026040500856_N58748_WithTracts.parquet")
# FILE_CONSDB = os.path.join(
#    DIR_VISITS, "constDbVisitTable-2025041500043-2026032500675_N85366_WithTracts.parquet"
# )
FILE_CONSDB = os.path.join(
    DIR_VISITS, "constDbVisitTable-2025041500043-2026040600288_N93006_WithTracts.parquet"
)

# ── Output directory ──────────────────────────────────────────────────────────
NB_TAG_OUT = "FINK_BLOCK_LC_03"
DIR_DATA_OUT = f"data_{NB_TAG_OUT}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)

print(f"Alert src file  : {FILE_SRC}  — exists: {os.path.exists(FILE_SRC)}")
print(f"Alert fp  file  : {FILE_FP}   — exists: {os.path.exists(FILE_FP)}")
print(f"Butler visit    : {FILE_BUTLER} — exists: {os.path.exists(FILE_BUTLER)}")
print(f"consDb  visit   : {FILE_CONSDB} — exists: {os.path.exists(FILE_CONSDB)}")
print(f"Output dir      : {os.path.abspath(DIR_DATA_OUT)}")

In [ ]:
# ── User choice: which visit table to join? ───────────────────────────────────
# Set VISIT_TABLE_CHOICE to "butler" or "consdb" (or run both — see Section 3)
VISIT_TABLE_CHOICE = "consdb"  # <-- change here: "butler" | "consdb"

# ── Join key column names ─────────────────────────────────────────────────────
# Alert side (same for src and fp)
COL_VISIT_ALERT = "r:visit"

# Visit table side
COL_VISIT_BUTLER = "id"  # Butler visitTable primary key
COL_VISIT_CONSDB = "visit_id"  # consDb visitTable primary key

print(f"Selected visit table : {VISIT_TABLE_CHOICE}")
print(f"Alert join key       : {COL_VISIT_ALERT}")
print(f"Butler join key      : {COL_VISIT_BUTLER}")
print(f"consDb  join key     : {COL_VISIT_CONSDB}")

## 1. Load all tables

In [ ]:
df_src = pd.read_parquet(FILE_SRC)
df_fp = pd.read_parquet(FILE_FP)

print("=" * 60)
print(f"src alerts  → {df_src.shape[0]:>8,} rows  × {df_src.shape[1]:>3} columns")
print(f"fp  alerts  → {df_fp.shape[0]:>8,} rows  × {df_fp.shape[1]:>3} columns")

In [ ]:
df_butler = pd.read_parquet(FILE_BUTLER)
df_consdb = pd.read_parquet(FILE_CONSDB)

print("=" * 60)
print(f"Butler visitTable → {df_butler.shape[0]:>8,} rows  × {df_butler.shape[1]:>3} columns")
print(f"consDb visitTable → {df_consdb.shape[0]:>8,} rows  × {df_consdb.shape[1]:>3} columns")

## 2. Schema inspection

In [ ]:
print("── src alert columns ──")
print(df_src.dtypes.to_string())
print()
print("── fp alert columns ──")
print(df_fp.dtypes.to_string())

In [ ]:
print("── Butler visitTable columns ──")
print(df_butler.dtypes.to_string())

In [ ]:
print("── consDb visitTable columns ──")
print(df_consdb.dtypes.to_string())

In [ ]:
# Quick preview
print("── src head ──")
display(df_src.head(3))
print("── fp head ──")
display(df_fp.head(3))

In [ ]:
print("── Butler visitTable head ──")
display(df_butler.head(3))
print("── consDb visitTable head ──")
display(df_consdb.head(3))

## 3. Verify the visit column exists in alert tables

In [ ]:
def detect_visit_col(df, candidates, label):
    """Return the first candidate column found in df, or None."""
    for c in candidates:
        if c in df.columns:
            print(f"  [{label}] found visit column: '{c}'")
            return c
    print(f"  [{label}] WARNING – none of {candidates} found in columns!")
    return None


alert_visit_candidates = ["r:visit", "visit", "visitId", "visit_id"]

print("Detecting visit column in alert tables …")
col_visit_src = detect_visit_col(df_src, alert_visit_candidates, "src")
col_visit_fp = detect_visit_col(df_fp, alert_visit_candidates, "fp")

print("\nDetecting visit column in Rubin tables …")
col_visit_butler = detect_visit_col(df_butler, ["id", "visitId", "visit", "visit_id"], "butler")
col_visit_consdb = detect_visit_col(df_consdb, ["visit_id", "visitId", "visit", "id"], "consdb")

print(f"\nSummary:")
print(f"  src    visit col : {col_visit_src}")
print(f"  fp     visit col : {col_visit_fp}")
print(f"  butler visit col : {col_visit_butler}")
print(f"  consdb visit col : {col_visit_consdb}")

## 4. Unique visit coverage

How many distinct visits appear in the alert tables vs. the Rubin visit tables?

In [ ]:
visits_src = set(df_src[col_visit_src].dropna().astype(int))
visits_fp = set(df_fp[col_visit_fp].dropna().astype(int))
visits_butler = set(df_butler[col_visit_butler].dropna().astype(int))
visits_consdb = set(df_consdb[col_visit_consdb].dropna().astype(int))

print(f"Unique visits in src alerts      : {len(visits_src):,}")
print(f"Unique visits in fp  alerts      : {len(visits_fp):,}")
print(f"Unique visits in Butler table    : {len(visits_butler):,}")
print(f"Unique visits in consDb table    : {len(visits_consdb):,}")
print()
print(f"src ∩ Butler : {len(visits_src & visits_butler):,}")
print(f"src ∩ consDb : {len(visits_src & visits_consdb):,}")
print(f"fp  ∩ Butler : {len(visits_fp & visits_butler):,}")
print(f"fp  ∩ consDb : {len(visits_fp & visits_consdb):,}")

In [ ]:
# Bar chart: visit coverage
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (alert_label, alert_visits) in zip(axes, [("src", visits_src), ("fp", visits_fp)]):
    common_butler = len(alert_visits & visits_butler)
    common_consdb = len(alert_visits & visits_consdb)
    only_alert = len(alert_visits - visits_butler - visits_consdb)
    labels = ["Only alert", "∩ Butler", "∩ consDb"]
    values = [only_alert, common_butler, common_consdb]
    colors = ["#95a5a6", "#4C9BE8", "#E8754C"]
    bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=1.4)
    for b, v in zip(bars, values):
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.5,
            f"{v:,}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    ax.set_title(f"{alert_label} alerts — visit coverage")
    ax.set_ylabel("Number of unique visits")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(os.path.join(DIR_DATA_OUT, "visit_coverage.png"), dpi=120)
plt.show()
print("Figure saved.")

## 5. Prepare the Rubin visit table for joining

We rename the join key to a common name (`visitId`) and cast to `int64`  
to ensure type compatibility with the alert `r:visit` column.

In [ ]:
def prepare_visit_table(df, visit_col, suffix):
    """
    Return a copy of a Rubin visit table ready for merging:
    - Rename visit_col → "visitId"
    - Cast visitId to int64
    - Add a suffix to all non-key columns to avoid name collisions
    """
    df = df.copy()
    df = df.rename(columns={visit_col: "visitId"})
    df["visitId"] = df["visitId"].astype("int64")
    # Rename all other columns with suffix
    rename_map = {c: f"{c}{suffix}" for c in df.columns if c != "visitId"}
    df = df.rename(columns=rename_map)
    return df


df_butler_ready = prepare_visit_table(df_butler, col_visit_butler, suffix="_butler")
df_consdb_ready = prepare_visit_table(df_consdb, col_visit_consdb, suffix="_consdb")

print(f"Butler (ready) : {df_butler_ready.shape[0]:,} rows × {df_butler_ready.shape[1]} cols")
print(f"consDb (ready) : {df_consdb_ready.shape[0]:,} rows × {df_consdb_ready.shape[1]} cols")
print("\nButler columns (first 10):", list(df_butler_ready.columns[:10]))
print("consDb columns (first 10):", list(df_consdb_ready.columns[:10]))

## 6. Prepare the alert tables for joining

Cast the `r:visit` column (or equivalent) to `int64`.

In [ ]:
def prepare_alerts(df, visit_col):
    """
    Return a copy of an alert DataFrame with the visit column cast to int64
    and renamed to 'visitId' for joining.
    The original column is kept under its original name as well.
    """
    df = df.copy()
    df["visitId"] = df[visit_col].astype("int64")
    return df


df_src_ready = prepare_alerts(df_src, col_visit_src)
df_fp_ready = prepare_alerts(df_fp, col_visit_fp)

print(f"src (ready) : {df_src_ready.shape[0]:,} rows × {df_src_ready.shape[1]} cols")
print(f"fp  (ready) : {df_fp_ready.shape[0]:,} rows × {df_fp_ready.shape[1]} cols")

## 7. Merge (left-join) alerts with Rubin visit tables

We perform a **left join**: every alert row is kept.  
If no matching visit is found in the Rubin table, the visit columns are `NaN`.

We produce four output tables:
1. `src` × Butler
2. `src` × consDb
3. `fp`  × Butler
4. `fp`  × consDb

In [ ]:
def merge_alert_visits(df_alert, df_visit, label_alert, label_visit):
    """
    Left-join df_alert with df_visit on 'visitId'.
    Reports match statistics.
    """
    merged = df_alert.merge(df_visit, on="visitId", how="left")

    # Count matched rows (where at least one visit column is not NaN)
    # Use the first non-key column of df_visit to detect matches
    visit_cols = [c for c in df_visit.columns if c != "visitId"]
    if visit_cols:
        n_matched = merged[visit_cols[0]].notna().sum()
        n_unmatched = merged[visit_cols[0]].isna().sum()
    else:
        n_matched = len(merged)
        n_unmatched = 0

    pct = 100 * n_matched / len(merged) if len(merged) > 0 else 0.0

    print(f"  [{label_alert} × {label_visit}]")
    print(f"    Input rows      : {len(df_alert):,}")
    print(f"    Output rows     : {len(merged):,}")
    print(f"    Matched         : {n_matched:,}  ({pct:.1f} %)")
    print(f"    Unmatched       : {n_unmatched:,}")
    print(f"    Output columns  : {merged.shape[1]}")
    return merged


print("Performing left joins …")
print()
df_src_butler = merge_alert_visits(df_src_ready, df_butler_ready, "src", "Butler")
print()
df_src_consdb = merge_alert_visits(df_src_ready, df_consdb_ready, "src", "consDb")
print()
df_fp_butler = merge_alert_visits(df_fp_ready, df_butler_ready, "fp", "Butler")
print()
df_fp_consdb = merge_alert_visits(df_fp_ready, df_consdb_ready, "fp", "consDb")

## 8. Inspect the joined tables

In [ ]:
print("── src × Butler (head) ──")
display(df_src_butler.head(3))

In [ ]:
print("── src × consDb (head) ──")
display(df_src_consdb.head(3))

In [ ]:
print("── fp × Butler (head) ──")
display(df_fp_butler.head(3))

In [ ]:
print("── fp × consDb (head) ──")
display(df_fp_consdb.head(3))

## 9. Summary statistics of the joined tables

In [ ]:
def join_summary(df, label, visit_source):
    """Print a compact summary of a joined DataFrame."""
    print(f"{'=' * 60}")
    print(f"{label} × {visit_source}")
    print(f"  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
    # Count columns from the visit table
    visit_sfx = "_butler" if visit_source == "Butler" else "_consdb"
    visit_cols = [c for c in df.columns if c.endswith(visit_sfx)]
    if visit_cols:
        n_matched = df[visit_cols[0]].notna().sum()
        print(f"  Matched rows   : {n_matched:,} / {len(df):,} ({100 * n_matched / len(df):.1f} %)")
    print(f"  Visit cols     : {len(visit_cols)} (suffix '{visit_sfx}')")
    # Band distribution if available
    band_col = next((c for c in df.columns if c in ["band", "r:band", "filter"]), None)
    if band_col:
        print(f"  Band distribution ({band_col}):")
        print(df[band_col].value_counts().to_string())


join_summary(df_src_butler, "src", "Butler")
join_summary(df_src_consdb, "src", "consDb")
join_summary(df_fp_butler, "fp", "Butler")
join_summary(df_fp_consdb, "fp", "consDb")

## 10. Explore key visit metadata columns from the joined table

We focus on the columns most useful for DRP processing:
- `tract`, `patch` (sky geometry)
- `band` / `filter` (photometric band)
- `detector` (CCD number)
- Observing time, seeing, sky brightness (if present in consDb)

In [ ]:
# Choose the primary joined table for exploration (consDb has more metadata)
df_explore = df_src_consdb.copy()
visit_sfx = "_consdb"

# Auto-detect interesting visit columns
interesting_keywords = [
    "tract",
    "patch",
    "band",
    "filter",
    "detector",
    "seeing",
    "sky",
    "zp",
    "airmass",
    "mjd",
    "exptime",
    "ra",
    "dec",
]
visit_cols = [c for c in df_explore.columns if c.endswith(visit_sfx)]
interesting_cols = [c for c in visit_cols if any(k in c.lower() for k in interesting_keywords)]

print(f"Visit columns with suffix '{visit_sfx}': {len(visit_cols)}")
print(f"Interesting visit metadata columns   : {len(interesting_cols)}")
for c in interesting_cols:
    print(f"  {c}")

In [ ]:
# Display a few key columns from the joined table
key_alert_cols = [
    c
    for c in ["r:diaObjectId", "r:visit", "r:detector", "r:band", "r:ra", "r:dec", "visitId", "group_label"]
    if c in df_explore.columns
]
key_visit_cols = interesting_cols[:8]  # show first 8
show_cols = key_alert_cols + key_visit_cols

print(f"Displaying {len(show_cols)} columns from src × consDb:")
display(df_explore[show_cols].head(10))

## 11. Tract / Patch distribution of matched alerts

In [ ]:
# Detect tract/patch columns
tract_col = next((c for c in df_src_consdb.columns if "tract" in c.lower() and c.endswith("_consdb")), None)
patch_col = next((c for c in df_src_consdb.columns if "patch" in c.lower() and c.endswith("_consdb")), None)

print(f"Tract column detected : {tract_col}")
print(f"Patch column detected : {patch_col}")

if tract_col:
    print(f"\nTract distribution (src × consDb):")
    print(df_src_consdb[tract_col].value_counts().head(20).to_string())

In [ ]:
if tract_col and patch_col:
    # Tract × Patch heatmap
    tp = df_src_consdb.groupby([tract_col, patch_col]).size().reset_index(name="n_alerts")
    pivot = tp.pivot_table(index=tract_col, columns=patch_col, values="n_alerts", fill_value=0)

    fig, ax = plt.subplots(figsize=(14, max(4, len(pivot) * 0.35)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel("Patch")
    ax.set_ylabel("Tract")
    ax.set_title("Number of src alerts per (tract, patch) — consDb join")
    plt.colorbar(im, ax=ax, label="N alerts")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_DATA_OUT, "tract_patch_heatmap_src_consdb.png"), dpi=120)
    plt.show()
    print("Figure saved.")

## 12. Save joined tables to `data_FINK_BLOCK_LC_03/`

In [ ]:
out_files = {
    "src_joined_butler.parquet": df_src_butler,
    "src_joined_consdb.parquet": df_src_consdb,
    "fp_joined_butler.parquet": df_fp_butler,
    "fp_joined_consdb.parquet": df_fp_consdb,
}

for fname, df in out_files.items():
    fpath = os.path.join(DIR_DATA_OUT, fname)
    df.to_parquet(fpath, index=False)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  Written: {fpath}  ({df.shape[0]:,} rows × {df.shape[1]} cols, {size_mb:.1f} MB)")

print("\nAll output files saved.")

## 13. Quick diagnostic: unmatched visits

Investigate which alert visits are NOT present in the chosen Rubin visit table.

In [ ]:
def unmatched_summary(df_alert, df_visit_ready, label_alert, label_visit):
    """
    Find alert visits not present in the Rubin visit table and report stats.
    """
    alert_visits = set(df_alert["visitId"].dropna().astype(int))
    visit_ids = set(df_visit_ready["visitId"].dropna().astype(int))
    missing = alert_visits - visit_ids
    pct = 100 * len(missing) / len(alert_visits) if alert_visits else 0.0
    print(f"  [{label_alert} × {label_visit}]")
    print(f"    Unique alert visits            : {len(alert_visits):,}")
    print(f"    Alert visits NOT in visit table: {len(missing):,} ({pct:.1f} %)")
    if missing:
        sample = sorted(missing)[:5]
        print(f"    Sample of missing visitIds: {sample}")


print("Unmatched visit analysis:")
unmatched_summary(df_src_ready, df_butler_ready, "src", "Butler")
unmatched_summary(df_src_ready, df_consdb_ready, "src", "consDb")
unmatched_summary(df_fp_ready, df_butler_ready, "fp", "Butler")
unmatched_summary(df_fp_ready, df_consdb_ready, "fp", "consDb")

## 14. (Optional) User-selected join: single enriched table

For convenience, expose the joined table selected by `VISIT_TABLE_CHOICE`.

In [ ]:
if VISIT_TABLE_CHOICE == "butler":
    df_src_enriched = df_src_butler
    df_fp_enriched = df_fp_butler
    visit_sfx_sel = "_butler"
else:
    df_src_enriched = df_src_consdb
    df_fp_enriched = df_fp_consdb
    visit_sfx_sel = "_consdb"

print(f"Selected visit table : {VISIT_TABLE_CHOICE}")
print(f"df_src_enriched : {df_src_enriched.shape[0]:,} rows × {df_src_enriched.shape[1]} cols")
print(f"df_fp_enriched  : {df_fp_enriched.shape[0]:,} rows × {df_fp_enriched.shape[1]} cols")

In [ ]:
# Summary of the enriched src table
print("Enriched src table — dtypes:")
print(df_src_enriched.dtypes.to_string())

In [ ]:
print("Enriched src table — head:")
display(df_src_enriched.head(5))

## Summary

| Output file | Rows | Columns | Join |
|-------------|------|---------|------|
| `src_joined_butler.parquet` | — | — | src × Butler |
| `src_joined_consdb.parquet` | — | — | src × consDb |
| `fp_joined_butler.parquet`  | — | — | fp  × Butler |
| `fp_joined_consdb.parquet`  | — | — | fp  × consDb |

All files are written to `data_FINK_BLOCK_LC_03/`.  
The join key is `visitId` (= `r:visit` in alerts, auto-detected in Rubin tables).  
Unmatched alert rows are retained with `NaN` in the visit columns (left join).


In [ ]:
df_src_consdb["dimm_seeing"].hist()